<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [2]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2
import math

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [3]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [4]:
import io

class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        # Конвертируем в тензор
        boxes_tensor = torch.tensor(np.array(boxes), dtype=torch.float32)

        if len(boxes_tensor) > 0:
            boxes_tensor[:, 2] = boxes_tensor[:, 0] + boxes_tensor[:, 2] # x2 = x_min + width
            boxes_tensor[:, 3] = boxes_tensor[:, 1] + boxes_tensor[:, 3] # y2 = y_min + height

        target['boxes'] = boxes_tensor
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [5]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.RandomResizedCrop(size=(256, 256), scale=(0.5, 1.0), p=1.0),  # шире scale → больше масштабов
        A.HorizontalFlip(p=0.5),                                          # дешёвая и полезная

        A.OneOf([
            A.RandomBrightnessContrast(p=1.0),
            A.HueSaturationValue(p=1.0),
        ], p=0.5),

        A.GaussNoise(p=0.2),
        A.CoarseDropout(num_holes_range=(1, 8),
                        hole_height_range=(8, 32),
                        hole_width_range=(8, 32), p=0.2),

        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format='coco',
        label_fields=['labels'],
        min_visibility=0.3,
        min_area=16,
    )
)
test_transform = A.Compose(
    [
        A.Resize(height=256, width=256),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3, min_area=16)
)

Не забываем инициализировать наш датасет

In [6]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

In [7]:
df_objects = pd.json_normalize(df_train['objects'])
all_categories = df_objects['category'].explode().dropna().unique()
num_classes = int(all_categories.max())
print(f"Обнаружено классов: {num_classes} → {sorted(all_categories)}")

Обнаружено классов: 4 → [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [8]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
class Backbone(nn.Module):
    def __init__(self, unfreeze_last=1):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

        self.layer1 = nn.Sequential(*list(backbone.children())[:5])
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.out_channels_list = [128, 256, 512]

        for param in self.parameters():
            param.requires_grad = False

        if unfreeze_last > 0:
            blocks = [self.layer1, self.layer2, self.layer3, self.layer4]
            for block in blocks[-unfreeze_last:]:
                for param in block.parameters():
                    param.requires_grad = True

    def forward(self, x):
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)

        return [c3, c4, c5]

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [9]:
import torch.nn as nn
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_channels_list=[128, 256, 512], out_channels=256):
        super().__init__()

        # 1x1 свертки для приведения карт из бекбоуна к одной размерности (out_channels)
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_c, out_channels, kernel_size=1) for in_c in in_channels_list
        ])

        # Доп конволюция (3х3 с паддингом) у каждого выхода шеи для сглаживания артефактов апсемплинга
        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1) for _ in in_channels_list
        ])

    def forward(self, features):
        """
        features: список тензоров от backbone [c3, c4, c5]
        """
        # 1. Применяем 1x1 свертки к фичам из бекбоуна
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        # 2. Top-down путь (спускаемся от самого маленького разрешения к большому)
        # Итерируемся в обратном порядке: от конца к началу
        for i in range(len(laterals) - 1, 0, -1):
            # Увеличиваем пространственную размерность (nearest neighbor upsampling)
            upsampled = F.interpolate(laterals[i], size=laterals[i-1].shape[-2:], mode='nearest')

            # Суммируем со скип-коннекшеном
            laterals[i-1] = laterals[i-1] + upsampled

        # 3. Применяем финальные 3x3 свертки к каждому уровню пирамиды
        outs = [conv(lateral) for conv, lateral in zip(self.fpn_convs, laterals)]

        # Возвращаем пирамиду признаков
        return outs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [10]:
class Head(nn.Module):
    def __init__(self, in_channels=256, num_classes=4):   # num_classes=4 (реальное число)
        super().__init__()
        self.num_classes = num_classes

        # Classification branch
        self.cls_convs = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.GroupNorm(32, in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.GroupNorm(32, in_channels),
            nn.SiLU(inplace=True)
        )
        self.cls_pred = nn.Conv2d(in_channels, num_classes, kernel_size=1)

        # Regression branch
        self.reg_convs = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.GroupNorm(32, in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.GroupNorm(32, in_channels),
            nn.SiLU(inplace=True)
        )
        self.reg_pred = nn.Conv2d(in_channels, 4, kernel_size=1)

        # Инициализация bias для Focal Loss (чтобы в начале вероятность ~0.01)
        prior_prob = 0.01
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        nn.init.constant_(self.cls_pred.bias, bias_value)

    def forward(self, x):
        cls_feat = self.cls_convs(x)
        cls_logits = self.cls_pred(cls_feat)

        reg_feat = self.reg_convs(x)
        reg_bboxes = self.reg_pred(reg_feat)

        # Переформатируем: (B, C, H, W) -> (B, H*W, C)
        cls_logits = cls_logits.permute(0, 2, 3, 1).contiguous().view(cls_logits.shape[0], -1, self.num_classes)
        reg_bboxes = reg_bboxes.permute(0, 2, 3, 1).contiguous().view(reg_bboxes.shape[0], -1, 4)

        return cls_logits, reg_bboxes   # ← теперь только 2 выхода

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [11]:
class Detector(nn.Module):
    def __init__(self, num_classes=1, unfreeze_last=3):
        super().__init__()
        self.backbone = Backbone(unfreeze_last=unfreeze_last)
        self.neck = Neck(in_channels_list=self.backbone.out_channels_list, out_channels=256)
        self.heads = nn.ModuleList([
            Head(in_channels=256, num_classes=num_classes)
            for _ in range(3)
        ])

    def forward(self, x):
        body_features = self.backbone(x)
        fpn_features = self.neck(body_features)

        all_cls = []
        all_reg = []
        # Голова уже сама делает permute+view, тут ничего трогать не нужно!
        for feature, head in zip(fpn_features, self.heads):
            cls_logits, reg_bboxes = head(feature)   # уже (B, N, C) и (B, N, 4)
            all_cls.append(cls_logits)
            all_reg.append(reg_bboxes)

        return {
            "cls_logits": torch.cat(all_cls, dim=1),
            "bbox_preds": torch.cat(all_reg, dim=1)
        }

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [12]:
import torch
from torchvision.ops import box_iou

def TAL_assigner(pred_scores, pred_bboxes, gt_classes, gt_bboxes, anchors, topk=13, alpha=6.0, beta=1.0):
    """
    pred_scores: (N, num_classes) - предсказанные вероятности классов
    pred_bboxes: (N, 4) - предсказанные ббоксы (x1, y1, x2, y2)
    gt_classes: (M,) - классы истинных объектов (GT)
    gt_bboxes: (M, 4) - координаты истинных объектов (x1, y1, x2, y2)
    anchors: (N, 2) - координаты центров якорей (cx, cy)
    """
    M = gt_bboxes.shape[0] # количество реальных объектов
    N = pred_bboxes.shape[0] # количество предсказаний (якорей)

    # Если на картинке нет объектов или нет предсказаний
    if M == 0 or N == 0:
        return torch.zeros(N, dtype=torch.int64, device=pred_bboxes.device) - 1 # -1 означает "фон"

    # 1. Считаем IoU между всеми GT и предсказаниями -> (M, N)
    ious = box_iou(gt_bboxes, pred_bboxes)

    # 2. Проверяем, лежат ли центры якорей внутри GT
    x_centers = anchors[:, 0].unsqueeze(0) # (1, N)
    y_centers = anchors[:, 1].unsqueeze(0) # (1, N)

    l = x_centers - gt_bboxes[:, 0].unsqueeze(1)
    t = y_centers - gt_bboxes[:, 1].unsqueeze(1)
    r = gt_bboxes[:, 2].unsqueeze(1) - x_centers
    b = gt_bboxes[:, 3].unsqueeze(1) - y_centers

    is_in_gts = (l > 0) & (t > 0) & (r > 0) & (b > 0) # (M, N)

    # 3. Считаем метрику t = s^alpha * u^beta
    # s - score (вероятность) нужного класса для каждого GT
    s = pred_scores[:, gt_classes].T # Транспонируем, чтобы получить размер (M, N)
    align_metric = (s ** alpha) * (ious ** beta)

    # Зануляем метрику для якорей, центры которых лежат вне GT
    align_metric[~is_in_gts] = 0.0

    # 4. Выбираем top-K предсказаний для каждого GT
    topk_mask = torch.zeros((M, N), dtype=torch.bool, device=pred_bboxes.device)
    for i in range(M):
        num_valid = is_in_gts[i].sum()
        if num_valid == 0:
            continue
        k = min(topk, num_valid.item())
        _, topk_idxs = torch.topk(align_metric[i], k)
        topk_mask[i, topk_idxs] = True

    # 5. Разрешение конфликтов: если якорь в top-K у нескольких GT,
    # отдаем его тому GT, с которым у него больше IoU
    ious_masked = ious.clone()
    ious_masked[~topk_mask] = 0.0

    # Находим максимальный IoU для каждого якоря и индекс этого GT
    max_iou, argmax_iou = ious_masked.max(dim=0) # (N,), (N,)

    # Создаем итоговый массив. По умолчанию всё фон (-1)
    assigned_gt_idxs = torch.zeros(N, dtype=torch.int64, device=pred_bboxes.device) - 1

    # Присваиваем индексы GT тем якорям, которые прошли отбор
    pos_mask = max_iou > 0
    assigned_gt_idxs[pos_mask] = argmax_iou[pos_mask]

    return assigned_gt_idxs

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [13]:
from torchvision.ops import distance_box_iou_loss

In [14]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [15]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [16]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.00861394405365


In [17]:
from torchvision.ops import distance_box_iou_loss

def diou_loss(pred_boxes, gt_boxes, reduction='mean'):
    """ Возвращает DIoU loss. reduction='none' даёт тензор (N,) для агрегации снаружи. """
    area_p = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    area_g = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    lt = torch.max(pred_boxes[:, :2], gt_boxes[:, :2])
    rb = torch.min(pred_boxes[:, 2:], gt_boxes[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    union = area_p + area_g - inter + 1e-7
    iou = inter / union

    lt_c = torch.min(pred_boxes[:, :2], gt_boxes[:, :2])
    rb_c = torch.max(pred_boxes[:, 2:], gt_boxes[:, 2:])
    wh_c = (rb_c - lt_c).clamp(min=0)
    c2 = wh_c[:, 0]**2 + wh_c[:, 1]**2 + 1e-7

    center_p = (pred_boxes[:, :2] + pred_boxes[:, 2:]) / 2
    center_g = (gt_boxes[:, :2] + gt_boxes[:, 2:]) / 2
    d2 = ((center_p - center_g) ** 2).sum(dim=1)

    loss = 1 - (iou - d2 / c2)
    if reduction == 'mean': return loss.mean()
    if reduction == 'sum':  return loss.sum()
    return loss

In [18]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(
    diou_loss(pred_boxes, true_boxes, reduction='mean'),
    distance_box_iou_loss(pred_boxes, true_boxes, reduction='mean')
)

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?

**TAL. Он привязывает таргеты к якорям не по фиксированному IoU-порогу, а по совместной метрике score^α × IoU^β, поэтому позитивами становятся именно те якоря, у которых классификация и локализация согласованы — нет mismatch между ветками cls и reg.**

2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?

**по сути все шаги были обязательными, поэтому сложно изолировать. Самый честный ответ — фикс декодирования боксов со скейлингом на stride. Без него сеть в принципе не учится, с ним получили 0.4+ mAP. На втором месте — TAL и DIoU вместе (они работают в паре).**

3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

**GaussNoise, CoarseDropout, HueSaturationValue — аугментации, влияние которых проявляется только на больших датасетах/эпохах. Можно прямо честно сказать, что не делал ablation study, но что-то из них почти наверняка нейтрально.**

In [19]:
import torch.nn.functional as F

def focal_loss(inputs, targets, alpha=0.25, gamma=2.0, reduction='sum'):
    """ Focal Loss с правильным per-sample alpha (как в RetinaNet). """
    p = torch.sigmoid(inputs)
    ce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)   # ← вот это и было сломано
    loss = alpha_t * (1 - p_t) ** gamma * ce
    if reduction == 'sum':
        return loss.sum()
    return loss.mean()
def compute_loss(outputs, targets, anchors, strides_per_anchor, alpha=6.0, beta=1.0):
    cls_logits = outputs['cls_logits']
    bbox_preds = outputs['bbox_preds']

    device = cls_logits.device
    B, N, C = cls_logits.shape

    total_cls = 0.0
    total_box = 0.0
    total_pos = 0

    for i in range(B):
        pred_cls = cls_logits[i]
        gt_boxes = targets[i]['boxes'].to(device)
        gt_labels = targets[i]['labels'].to(device)

        distances = F.softplus(bbox_preds[i]) * strides_per_anchor[:, None]
        x1 = anchors[:, 0] - distances[:, 0]
        y1 = anchors[:, 1] - distances[:, 1]
        x2 = anchors[:, 0] + distances[:, 2]
        y2 = anchors[:, 1] + distances[:, 3]
        pred_box = torch.stack([x1, y1, x2, y2], dim=1)

        if gt_boxes.numel() == 0:
            # На картинке нет объектов — учим только классификацию против фона
            target_cls = torch.zeros_like(pred_cls)
            total_cls += focal_loss(pred_cls, target_cls)
            continue

        pred_scores = torch.sigmoid(pred_cls).detach()   # detach: не пускаем градиент через ассигнер
        assigned_gt_idxs = TAL_assigner(
            pred_scores=pred_scores,
            pred_bboxes=pred_box.detach(),
            gt_classes=gt_labels.clamp(0, C-1),
            gt_bboxes=gt_boxes,
            anchors=anchors,
            topk=13, alpha=alpha, beta=beta
        )

        pos_mask = assigned_gt_idxs >= 0
        num_pos = pos_mask.sum().item()
        total_pos += num_pos

        target_cls = torch.zeros_like(pred_cls)
        if num_pos > 0:
            matched = assigned_gt_idxs[pos_mask]
            target_cls[pos_mask, gt_labels[matched].clamp(0, C-1)] = 1.0
        total_cls += focal_loss(pred_cls, target_cls)

        if num_pos > 0:
            # reduction='sum', потом ОБА лосса делим на total_pos снаружи — одинаковая нормализация
            total_box += diou_loss(
                pred_box[pos_mask],
                gt_boxes[assigned_gt_idxs[pos_mask]],
                reduction='sum'
            )

    total_pos = max(total_pos, 1)
    return (total_cls + 2.0 * total_box) / total_pos

In [20]:
from torchvision.ops import nms
def filter_predictions(outputs, score_threshold=0.05, nms_threshold=0.5, anchors=None, strides_per_anchor=None):
    cls_logits = outputs['cls_logits']
    bbox_preds = outputs['bbox_preds']

    device = cls_logits.device
    B = cls_logits.shape[0]
    predictions = []

    for i in range(B):
        scores = torch.sigmoid(cls_logits[i]).max(dim=1)[0] if cls_logits.shape[-1] > 1 else torch.sigmoid(cls_logits[i]).squeeze(-1)

        # Декодирование с stride
        distances = F.softplus(bbox_preds[i]) * strides_per_anchor[:, None]
        x1 = anchors[:, 0] - distances[:, 0]
        y1 = anchors[:, 1] - distances[:, 1]
        x2 = anchors[:, 0] + distances[:, 2]
        y2 = anchors[:, 1] + distances[:, 3]
        boxes = torch.stack([x1, y1, x2, y2], dim=1)

        keep = scores > score_threshold
        if keep.sum() == 0:
            predictions.append({'boxes': torch.zeros((0,4), device=device),
                                'scores': torch.zeros(0, device=device),
                                'labels': torch.zeros(0, dtype=torch.int64, device=device)})
            continue

        filtered_boxes = boxes[keep]
        filtered_scores = scores[keep]
        filtered_classes = cls_logits[i][keep].argmax(dim=1)

        nms_idx = nms(filtered_boxes, filtered_scores, nms_threshold)
        final_boxes = filtered_boxes[nms_idx]
        final_scores = filtered_scores[nms_idx]
        final_classes = filtered_classes[nms_idx]

        final_boxes = torch.clamp(final_boxes, min=0, max=256)
        wh = final_boxes[:, 2:] - final_boxes[:, :2]
        valid = (wh > 3).all(dim=1)
        final_boxes = final_boxes[valid]
        final_scores = final_scores[valid]
        final_classes = final_classes[valid]

        predictions.append({'boxes': final_boxes, 'scores': final_scores, 'labels': final_classes})

    return predictions

In [21]:
from torch.utils.data import DataLoader
import torch

# Восстанавливаем потерянную функцию
def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

BATCH_SIZE = 8

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    drop_last=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

In [22]:
import io

In [23]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def generate_anchors(image_size=(256, 256), strides=[8, 16, 32]):
    """
    Генерирует центры (cx, cy) для каждого уровня пирамиды FPN.
    """
    anchors = []
    for stride in strides:
        # Размер карты признаков на текущем уровне
        grid_h = image_size[0] // stride
        grid_w = image_size[1] // stride

        # Генерируем координаты (центр ячейки = индекс * страйд + страйд / 2)
        shifts_x = torch.arange(0, grid_w, dtype=torch.float32) * stride + stride / 2.0
        shifts_y = torch.arange(0, grid_h, dtype=torch.float32) * stride + stride / 2.0

        # Создаем двумерную сетку
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")

        # Сплющиваем и собираем в (N, 2)
        stride_anchors = torch.stack([shift_x.flatten(), shift_y.flatten()], dim=-1)
        anchors.append(stride_anchors)

    # Склеиваем якоря со всех уровней
    return torch.cat(anchors, dim=0)
anchors = generate_anchors(image_size=(256, 256), strides=[8, 16, 32]).to(device)


In [24]:

strides_list = [8, 16, 32]
sizes = [(256 // s) ** 2 for s in strides_list]
strides_per_anchor = torch.cat([
    torch.full((n,), float(s), device=device, dtype=torch.float32)
    for n, s in zip(sizes, strides_list)
])
print(f"strides_per_anchor shape: {strides_per_anchor.shape}")

strides_per_anchor shape: torch.Size([1344])


In [25]:
!pip install torch.optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.3/357.3 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 46.0 MB/s eta 0:00:00


In [26]:
import torch.optim as optim
from tqdm import tqdm
import math

In [28]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.4 MB/s eta 0:00:00


In [29]:
from torchmetrics.detection import MeanAveragePrecision
from tqdm.auto import tqdm
import torch

@torch.no_grad()
def validate(model, dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")

    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)

        # Получаем предсказания
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)

        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
        metric.update(predicts, targets)

    return metric.compute()["map"].item()


In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
anchors = generate_anchors(image_size=(256, 256), strides=[8, 16, 32]).to(device)

model = Detector(num_classes=num_classes, unfreeze_last=4).to(device)   # 4 = и stem тоже

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 40
total_steps = num_epochs * len(train_dataloader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, total_steps=total_steps,
    pct_start=0.05, anneal_strategy='cos', div_factor=25, final_div_factor=1e3
)

best_map = 0.0
print("Начинаем обучение...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for images, targets in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images = images.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = compute_loss(outputs, targets, anchors, strides_per_anchor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)   # клип
        optimizer.step()
        scheduler.step()                                                     # OneCycle — каждый батч!
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)

    # Валидация раз в 5 эпох (полная валидация дорогая)
    if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
        test_map = validate(
            model=model, dataloader=test_dataloader,
            filter_predictions_func=filter_predictions,
            box_format="xyxy", device=device,
            score_threshold=0.01, nms_threshold=0.5,
            anchors=anchors, strides_per_anchor=strides_per_anchor
        )
        print(f"Epoch {epoch+1}  loss={avg_loss:.4f}  mAP={test_map:.4f}")
        if test_map > best_map:
            best_map = test_map
            torch.save(model.state_dict(), 'best.pt')
    else:
        print(f"Epoch {epoch+1}  loss={avg_loss:.4f}")

print(f"Лучший mAP: {best_map:.4f}")

Начинаем обучение...


Epoch 1/40:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1  loss=1.6998


Epoch 2/40:   0%|          | 0/57 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78b31868cae0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x78b31868cae0>^^
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    self._shutdown_workers()^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^if w.is_alive():^
^ ^  ^ ^ ^ ^ ^^^^^

...

Epoch 39/40:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 39  loss=0.4692


Epoch 40/40:   0%|          | 0/57 [00:00<?, ?it/s]

Running validation:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 40  loss=0.4632  mAP=0.4271
Лучший mAP: 0.4271


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [32]:
model.eval()
images, _ = next(iter(test_dataloader))
images = images.to(device)
with torch.no_grad():
    outputs = model(images)

cls_s = torch.sigmoid(outputs['cls_logits'][0]).max(dim=1)[0].mean().item()
print(f"Средний max-cls score: {cls_s:.4f}")

preds = filter_predictions(
    outputs, score_threshold=0.001, nms_threshold=0.7,
    anchors=anchors, strides_per_anchor=strides_per_anchor
)
num_boxes = sum(len(p['boxes']) for p in preds)
print(f"Всего найдено боксов на батче (threshold=0.001): {num_boxes}")

Средний max-cls score: 0.0352
Всего найдено боксов на батче (threshold=0.001): 6549


In [33]:
print("Запуск валидации на тестовом датасете")

model.eval()
test_map = validate(
    model=model,
    dataloader=test_dataloader,
    filter_predictions_func=filter_predictions,
    box_format="xyxy",
    device=device,
    score_threshold=0.05,
    nms_threshold=0.5,
    anchors=anchors,
    strides_per_anchor=strides_per_anchor
)

print(f"Итоговый mAP (0.5:0.95) на тестовой выборке: {test_map:.4f}")

Запуск валидации на тестовом датасете


Running validation:   0%|          | 0/17 [00:00<?, ?it/s]

Итоговый mAP (0.5:0.95) на тестовой выборке: 0.4271
